# Stack Overflow Question Quality Classifier

**CSE 575 – Statistical Machine Learning** · Project area **P11: Learning from CQA data**

End-to-end ML/NLP pipeline that turns the Stack Overflow questions CSV into a binary **question-quality classifier**.

**Research question:** Can textual and structural features of a Stack Overflow question predict community-judged quality?

### Pipeline
1. Load & explore the dataset  
2. Preprocess (dedupe, HTML parse, age filter)  
3. Define quality labels (score threshold, no leakage)  
4. Extract text (TF-IDF) + structural features  
5. Train Logistic Regression, Naive Bayes, Random Forest  
6. Evaluate on a held-out test set  
7. Compare feature sets and interpret results  

### How to run
1. Select the project `.venv` kernel (created in this repo).  
2. Confirm `data/stackoverflow_questions_2023plus.csv` exists.  
3. **Run All**. Expect ~10–25 minutes (Random Forest tuning dominates).


In [1]:
# 0. Setup
import re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay, classification_report,
)

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 80)

# Paths: notebook lives at repo root; CSV is under data/
DATA_PATH = Path("data/stackoverflow_questions_2023plus.csv")
if not DATA_PATH.exists():
    # fallbacks if the notebook is opened from docs/ or elsewhere
    for alt in [
        Path("stackoverflow_questions_2023plus.csv"),
        Path("../CSE-575_SML_Project/data/stackoverflow_questions_2023plus.csv"),
        Path("stackoverflow-cqa-dataset/stackoverflow_questions_2023plus.csv"),
        Path("/Users/nithinreddy/Desktop/docs/CSE-575_SML_Project/data/stackoverflow_questions_2023plus.csv"),
        Path("/Users/nithinreddy/Desktop/docs/stackoverflow-cqa-dataset/stackoverflow_questions_2023plus.csv"),
    ]:
        if alt.exists():
            DATA_PATH = alt
            break

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
MIN_AGE_DAYS = 30
HIGH_Q_THRESHOLD = 1  # Score >= 1 -> high quality

FIG_DIR, RES_DIR = Path("figures"), Path("results")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_STATE)

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print("DATA_PATH =", DATA_PATH.resolve())
assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"

DATA_PATH = /Users/nithinreddy/Desktop/CSE-575_SML_Project/data/stackoverflow_questions_2023plus.csv


## 1. Data collection & exploration

Each row is one Stack Overflow **question** with title, HTML body, tags, community feedback, and asker reputation.


In [2]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=["CreationDate", "LastEditDate"])
print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw.CreationDate.min()} -> {df_raw.CreationDate.max()}")
df_raw.head(3)

Shape: (10000, 14)
Date range: 2026-04-01 12:49:50 -> 2026-10-27 21:51:58


,QuestionId,Title,Body,Tags,Score,ViewCount,AnswerCount,CommentCount,FavoriteCount,CreationDate,LastEditDate,AcceptedAnswerId,UserId,UserReputation
0,80012966,Many wordpress editing aspects affecting since PHP issue,<p>I'm just wondering if anyone can tell me what exactly is causing my WordP...,|php|wordpress|plugins|,-1,3,0,0,0,2026-10-27 21:51:58,NaT,NaN,33241767.0,1.0
1,80012962,How do I do a proper chase camera without breaking culling?,"<p>In ThreeJS and Chrome, with a perspective camera designed to follow my ch...",|three.js|camera|culling|,0,37,0,0,0,2026-10-27 20:22:27,NaT,NaN,3851244.0,198.0
2,80012958,Tkinter Toplevel Windows Persisting in Memory When Deleted,<p>I'm working on a notepad program for Python and Tkinter.</p>\n<p>Whenever...,|python|tkinter|garbage-collection|toplevel|,0,30,0,0,0,2026-10-27 19:50:42,NaT,NaN,33240080.0,1.0


In [3]:
info = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_%": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(),
})
display(info)
df_raw[["Score", "ViewCount", "AnswerCount", "CommentCount", "FavoriteCount", "UserReputation"]].describe().round(2)

,dtype,missing,missing_%,n_unique
QuestionId,int64,0,0.00,10000
Title,str,0,0.00,10000
Body,str,0,0.00,10000
Tags,str,0,0.00,8655
Score,int64,0,0.00,42
ViewCount,int64,0,0.00,685
AnswerCount,int64,0,0.00,15
CommentCount,int64,0,0.00,35
FavoriteCount,int64,0,0.00,1
CreationDate,datetime64[us],0,0.00,9995


,Score,ViewCount,AnswerCount,CommentCount,FavoriteCount,UserReputation
count,10000.00,10000.00,10000.00,10000.00,10000.0,9911.00
mean,0.39,178.11,1.23,2.34,0.0,4430.89
std,2.45,337.33,1.23,3.67,0.0,26019.09
min,-14.00,3.00,0.00,0.00,0.0,1.00
25%,0.00,97.00,1.00,0.00,0.0,11.00
50%,0.00,136.00,1.00,1.00,0.0,69.00
75%,1.00,192.00,2.00,3.00,0.0,754.50
max,37.00,24100.00,15.00,50.00,0.0,892007.00


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(df_raw.Score.clip(-10, 15), discrete=True, ax=axes[0], color="#4C72B0")
axes[0].set(title="Score distribution (clipped [-10, 15])", xlabel="Score", ylabel="Questions")

bucket = pd.cut(df_raw.Score, [-np.inf, -1, 0, np.inf], labels=["Negative (<0)", "Zero", "Positive (>0)"])
bucket.value_counts().reindex(["Negative (<0)", "Zero", "Positive (>0)"]).plot.bar(
    ax=axes[1], color=["#C44E52", "#8C8C8C", "#55A868"])
axes[1].set(title="Score sign", xlabel="", ylabel="Questions")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); savefig("01_score_distribution"); plt.show()
print(bucket.value_counts(normalize=True).round(3))

Score
Zero             0.438
Positive (>0)    0.376
Negative (<0)    0.186
Name: proportion, dtype: float64


/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/436193380.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("01_score_distribution"); plt.show()


## 2. Preprocessing

1. Drop duplicate / empty questions  
2. Drop questions younger than 30 days (unstable scores)  
3. Parse HTML body → prose text, code text, structure counts


In [5]:
df = df_raw.copy()
n0 = len(df)
df = df.drop_duplicates(subset="QuestionId")
df = df.drop_duplicates(subset=["Title", "Body"])
n_dup = n0 - len(df)
df = df.dropna(subset=["Title", "Body"])
df = df[(df.Title.str.strip() != "") & (df.Body.str.strip() != "")]
n_missing = n0 - n_dup - len(df)

snapshot = df.CreationDate.max()
cutoff = snapshot - pd.Timedelta(days=MIN_AGE_DAYS)
n_before_age = len(df)
df = df[df.CreationDate <= cutoff].reset_index(drop=True)

print(f"Original: {n0}")
print(f"Removed duplicates: {n_dup}")
print(f"Removed empty title/body: {n_missing}")
print(f"Removed (<{MIN_AGE_DAYS}d old): {n_before_age - len(df)}  (snapshot={snapshot.date()}, cutoff={cutoff.date()})")
print(f"Kept: {len(df)}")

Original: 10000
Removed duplicates: 0
Removed empty title/body: 0
Removed (<30d old): 989  (snapshot=2026-10-27, cutoff=2026-09-27)
Kept: 9011


In [6]:
WS = re.compile(r"\s+")

def parse_body(html: str) -> dict:
    soup = BeautifulSoup(html if isinstance(html, str) else "", "html.parser")
    pre_blocks = soup.find_all("pre")
    code_text = "\n".join(p.get_text() for p in pre_blocks)
    n_code_lines = sum(len([l for l in p.get_text().splitlines() if l.strip()]) for p in pre_blocks)
    n_inline_code = sum(1 for c in soup.find_all("code") if c.find_parent("pre") is None)
    feats = {
        "n_code_blocks": len(pre_blocks),
        "n_code_lines": n_code_lines,
        "n_inline_code": n_inline_code,
        "n_links": len(soup.find_all("a")),
        "n_images": len(soup.find_all("img")),
        "n_paragraphs": len(soup.find_all("p")),
        "n_list_items": len(soup.find_all("li")),
        "n_blockquotes": len(soup.find_all("blockquote")),
        "n_headers": len(soup.find_all(re.compile(r"^h[1-6]$"))),
    }
    for p in pre_blocks:
        p.decompose()
    feats["body_text"] = WS.sub(" ", soup.get_text(" ")).strip()
    feats["code_text"] = code_text
    return feats

t0 = time.time()
parsed = pd.DataFrame([parse_body(h) for h in df.Body])
df = pd.concat([df, parsed], axis=1)
print(f"Parsed {len(df)} bodies in {time.time() - t0:.1f}s")
df[["Title", "body_text", "n_code_blocks", "n_code_lines", "n_links"]].head(3)

Parsed 9011 bodies in 3.3s


,Title,body_text,n_code_blocks,n_code_lines,n_links
0,How to associate static information for cond forms?,"I'm trying to use Rhombus static in my program, and there was only one place...",1,14,0
1,How to refactor hardcoded coordinate offsets when drawing custom composite s...,"I am drawing a custom ""stick figure"" shape in a C# WinForms application. Cur...",1,18,0
2,Why is this loop unroll on the avx512 pagecpy I made faster than before?,I have been trying to create a memcpy and pagecpy implementation that is rou...,2,62,2


## 3. Quality label (no leakage)

$$
y = \begin{cases}
1 \text{ (high)} & \text{if Score} \ge 1 \\
0 \text{ (low)} & \text{if Score} \le 0
\end{cases}
$$

**Excluded from features** (post-hoc / leakage): `Score`, `ViewCount`, `AnswerCount`, `CommentCount`, `FavoriteCount`, `AcceptedAnswerId`, `LastEditDate`, `UserReputation`, ids, dates.


In [7]:
df["label"] = (df.Score >= HIGH_Q_THRESHOLD).astype(int)
counts = df.label.value_counts().rename({0: "Low (Score ≤ 0)", 1: "High (Score ≥ 1)"})
display(pd.DataFrame({"count": counts, "share": (counts / counts.sum()).round(3)}))

ax = counts.plot.bar(color=["#C44E52", "#55A868"], figsize=(6, 3.5))
ax.set(title="Class balance", ylabel="Questions"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); savefig("02_class_balance"); plt.show()

,count,share
label,,
Low (Score ≤ 0),5602,0.622
High (Score ≥ 1),3409,0.378


/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/1422475859.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("02_class_balance"); plt.show()


## 4. Feature extraction

- **Text:** TF-IDF on title and body prose (unigrams + bigrams)  
- **Structural:** lengths, code/links/lists, tag indicators, style signals


In [8]:
WH_START = re.compile(
    r"^\s*(how|what|why|when|where|which|who|is|are|can|could|should|does|do|did|will|would)\b", re.I)
ERROR_RE = re.compile(
    r"\b(?:error|exception|traceback|stack ?trace|fails?|failed|crash(?:es|ed)?|warning)\b", re.I)
SENT_SPLIT = re.compile(r"[.!?]+\s")

def split_tags(s: str):
    return [t for t in str(s).split("|") if t]

df["tags_list"] = df.Tags.apply(split_tags)
df["tags_str"] = df.tags_list.apply(" ".join)
df["title_chars"] = df.Title.str.len()
df["title_words"] = df.Title.str.split().str.len()
df["title_has_qmark"] = df.Title.str.contains(r"\?").astype(int)
df["title_starts_wh"] = df.Title.str.match(WH_START).astype(int)
df["body_chars"] = df.body_text.str.len()
df["body_words"] = df.body_text.str.split().str.len().fillna(0)
df["has_code"] = (df.n_code_blocks > 0).astype(int)
df["code_ratio"] = df.code_text.str.len() / (df.code_text.str.len() + df.body_chars + 1)
df["n_tags"] = df.tags_list.str.len()
df["avg_sentence_len"] = df.body_text.apply(lambda t: len(t.split()) / max(1, len(SENT_SPLIT.split(t))))
df["upper_ratio"] = df.body_text.apply(
    lambda t: sum(c.isupper() for c in t) / max(1, sum(c.isalpha() for c in t)))
df["n_exclaim"] = df.body_text.str.count("!")
df["mentions_error"] = df.body_text.str.contains(ERROR_RE).astype(int)

NUMERIC_FEATURES = [
    "title_chars", "title_words", "title_has_qmark", "title_starts_wh",
    "body_chars", "body_words", "n_code_blocks", "n_code_lines", "n_inline_code",
    "has_code", "code_ratio", "n_links", "n_images", "n_paragraphs", "n_list_items",
    "n_blockquotes", "n_headers", "n_tags", "avg_sentence_len", "upper_ratio",
    "n_exclaim", "mentions_error",
]
TEXT_TITLE, TEXT_BODY, TAGS_COL = "Title", "body_text", "tags_str"
df[NUMERIC_FEATURES].describe().T.round(2).head(12)

,count,mean,std,min,25%,50%,75%,max
title_chars,9011.0,69.34,26.63,15.0,49.0,66.00,86.00,150.0
title_words,9011.0,10.93,4.39,2.0,8.0,10.00,14.00,30.0
title_has_qmark,9011.0,0.36,0.48,0.0,0.0,0.00,1.00,1.0
title_starts_wh,9011.0,0.37,0.48,0.0,0.0,0.00,1.00,1.0
body_chars,9011.0,845.79,663.19,0.0,420.0,676.00,1062.00,11661.0
body_words,9011.0,141.31,105.91,0.0,73.0,115.00,177.00,2028.0
n_code_blocks,9011.0,1.72,1.78,0.0,0.0,1.00,2.50,18.0
n_code_lines,9011.0,29.45,55.12,0.0,0.0,13.00,34.00,736.0
n_inline_code,9011.0,3.18,5.25,0.0,0.0,1.00,4.00,59.0
has_code,9011.0,0.73,0.44,0.0,0.0,1.00,1.00,1.0


In [9]:
plot_feats = ["title_words", "body_words", "n_code_lines", "n_paragraphs", "n_links", "n_tags"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, f in zip(axes.ravel(), plot_feats):
    sns.boxplot(data=df, x="label", y=np.log1p(df[f]), ax=ax, showfliers=False,
                hue="label", palette=["#C44E52", "#55A868"], legend=False)
    ax.set(title=f"log(1+{f})", xlabel="")
    ax.set_xticks([0, 1], ["Low", "High"])
plt.suptitle("Structural features by quality class", y=1.01)
plt.tight_layout(); savefig("03_structural_by_class"); plt.show()

/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/2682989827.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("03_structural_by_class"); plt.show()


## 5. Model development

Stratified 80/20 split. Tune with 5-fold CV on train only. Feature sets: **text**, **structural**, **combined**.


In [10]:
FEATURE_COLS = [TEXT_TITLE, TEXT_BODY, TAGS_COL] + NUMERIC_FEATURES
X = df[FEATURE_COLS]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"High-quality share  train={y_train.mean():.3f}  test={y_test.mean():.3f}")
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

Train: 7208 | Test: 1803
High-quality share  train=0.378  test=0.378


In [11]:
def tfidf_title():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True,
                           stop_words="english", max_features=5_000)

def tfidf_body():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True,
                           stop_words="english", max_features=20_000)

def tag_vectorizer():
    return CountVectorizer(token_pattern=r"\S+", binary=True, min_df=20, lowercase=False)

def numeric_pipe(scaler="standard"):
    scalers = {"standard": StandardScaler(), "minmax": MinMaxScaler(clip=True), None: "passthrough"}
    return Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", scalers[scaler]),
    ])

def make_features(feature_set, numeric_scaler="standard", dense=False):
    parts = []
    if feature_set in ("text", "combined"):
        parts += [("title_tfidf", tfidf_title(), TEXT_TITLE),
                  ("body_tfidf", tfidf_body(), TEXT_BODY)]
    if feature_set in ("structural", "combined"):
        parts += [("numeric", numeric_pipe(numeric_scaler), NUMERIC_FEATURES),
                  ("tags", tag_vectorizer(), TAGS_COL)]
    return ColumnTransformer(parts, sparse_threshold=0.0 if dense else 0.3)

def get_experiments():
    exps = {}
    for fs in ["text", "structural", "combined"]:
        exps[("Logistic Regression", fs)] = (
            Pipeline([("features", make_features(fs, "standard")),
                      ("clf", LogisticRegression(max_iter=3000, class_weight="balanced",
                                                 random_state=RANDOM_STATE))]),
            {"clf__C": [0.01, 0.1, 0.5, 1.0, 5.0]},
        )
        if fs == "structural":
            exps[("Naive Bayes", fs)] = (
                Pipeline([("features", make_features(fs, "standard", dense=True)),
                          ("clf", GaussianNB())]),
                {"clf__var_smoothing": [1e-9, 1e-6, 1e-3, 1e-1]},
            )
        else:
            exps[("Naive Bayes", fs)] = (
                Pipeline([("features", make_features(fs, "minmax")),
                          ("clf", MultinomialNB())]),
                {"clf__alpha": [0.01, 0.1, 0.5, 1.0], "clf__fit_prior": [True, False]},
            )
        exps[("Random Forest", fs)] = (
            Pipeline([("features", make_features(fs, None)),
                      ("clf", RandomForestClassifier(
                          n_estimators=200, class_weight="balanced_subsample",
                          n_jobs=-1, random_state=RANDOM_STATE))]),
            {"clf__max_depth": [20, None],
             "clf__min_samples_leaf": [1, 5],
             "clf__max_features": ["sqrt"]},
        )
    return exps

experiments = get_experiments()
print(f"{len(experiments)} model × feature-set experiments")

9 model × feature-set experiments


In [12]:
fitted, cv_rows = {}, []
for (model_name, fs), (pipe, grid) in experiments.items():
    t0 = time.time()
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    fitted[(model_name, fs)] = gs.best_estimator_
    cv_rows.append({
        "Model": model_name, "Features": fs,
        "CV F1 (mean)": gs.best_score_,
        "CV F1 (std)": gs.cv_results_["std_test_score"][gs.best_index_],
        "Best params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()},
        "Time (s)": round(time.time() - t0, 1),
    })
    print(f"{model_name:20s} | {fs:10s} | CV F1={gs.best_score_:.4f} | {gs.best_params_} | {time.time()-t0:.1f}s")

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(RES_DIR / "cv_results.csv", index=False)
cv_results.round(4)

Logistic Regression  | text       | CV F1=0.5511 | {'clf__C': 0.01} | 9.2s


Naive Bayes          | text       | CV F1=0.5197 | {'clf__alpha': 0.1, 'clf__fit_prior': False} | 10.2s


Random Forest        | text       | CV F1=0.5203 | {'clf__max_depth': 20, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 5} | 13.2s


Logistic Regression  | structural | CV F1=0.6654 | {'clf__C': 0.1} | 0.4s


Naive Bayes          | structural | CV F1=0.6164 | {'clf__var_smoothing': 0.1} | 0.3s


Random Forest        | structural | CV F1=0.6492 | {'clf__max_depth': 20, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 5} | 5.6s


Logistic Regression  | combined   | CV F1=0.6688 | {'clf__C': 0.1} | 9.2s


Naive Bayes          | combined   | CV F1=0.6009 | {'clf__alpha': 0.1, 'clf__fit_prior': False} | 10.8s


Random Forest        | combined   | CV F1=0.6298 | {'clf__max_depth': 20, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 5} | 13.6s


,Model,Features,CV F1 (mean),CV F1 (std),Best params,Time (s)
0,Logistic Regression,text,0.5511,0.0199,{'C': 0.01},9.2
1,Naive Bayes,text,0.5197,0.0183,"{'alpha': 0.1, 'fit_prior': False}",10.2
2,Random Forest,text,0.5203,0.0203,"{'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 5}",13.2
3,Logistic Regression,structural,0.6654,0.0090,{'C': 0.1},0.4
4,Naive Bayes,structural,0.6164,0.0143,{'var_smoothing': 0.1},0.3
5,Random Forest,structural,0.6492,0.0101,"{'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 5}",5.6
6,Logistic Regression,combined,0.6688,0.0114,{'C': 0.1},9.2
7,Naive Bayes,combined,0.6009,0.0174,"{'alpha': 0.1, 'fit_prior': False}",10.8
8,Random Forest,combined,0.6298,0.0103,"{'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 5}",13.6


## 6. Evaluation on held-out test set


In [13]:
def evaluate(model, X_, y_):
    pred = model.predict(X_)
    proba = model.predict_proba(X_)[:, 1]
    return {
        "Accuracy": accuracy_score(y_, pred),
        "Precision": precision_score(y_, pred, zero_division=0),
        "Recall": recall_score(y_, pred),
        "F1": f1_score(y_, pred),
        "Macro F1": f1_score(y_, pred, average="macro"),
        "ROC-AUC": roc_auc_score(y_, proba),
    }

rows = []
for strategy in ["most_frequent", "stratified"]:
    dummy = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({"Model": f"Dummy ({strategy})", "Features": "–", **evaluate(dummy, X_test, y_test)})
for (model_name, fs), model in fitted.items():
    rows.append({"Model": model_name, "Features": fs, **evaluate(model, X_test, y_test)})

results = pd.DataFrame(rows)
results.to_csv(RES_DIR / "test_results.csv", index=False)
(results.style.format({c: "{:.3f}" for c in results.columns[2:]})
        .background_gradient(subset=["F1", "Macro F1", "ROC-AUC"], cmap="Greens"))

,Model,Features,Accuracy,Precision,Recall,F1,Macro F1,ROC-AUC
0,Dummy (most_frequent),–,0.622,0.000,0.000,0.000,0.383,0.500
1,Dummy (stratified),–,0.516,0.362,0.365,0.364,0.487,0.487
2,Logistic Regression,text,0.624,0.503,0.562,0.530,0.608,0.674
3,Naive Bayes,text,0.615,0.491,0.523,0.507,0.595,0.649
4,Random Forest,text,0.642,0.530,0.479,0.503,0.612,0.667
5,Logistic Regression,structural,0.676,0.547,0.837,0.662,0.676,0.791
6,Naive Bayes,structural,0.659,0.539,0.685,0.603,0.653,0.721
7,Random Forest,structural,0.673,0.546,0.801,0.649,0.671,0.765
8,Logistic Regression,combined,0.691,0.562,0.827,0.669,0.690,0.795
9,Naive Bayes,combined,0.664,0.551,0.610,0.579,0.650,0.732


In [14]:
plot_df = results[~results.Model.str.startswith("Dummy")].melt(
    id_vars=["Model", "Features"],
    value_vars=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    var_name="Metric", value_name="Value")
g = sns.catplot(data=plot_df, x="Features", y="Value", hue="Model", col="Metric",
                kind="bar", col_wrap=5, height=3.6, aspect=0.8, palette="Set2",
                order=["text", "structural", "combined"])
g.set(ylim=(0, 1)); g.set_titles("{col_name}")
dummy_f1 = results.loc[results.Model == "Dummy (stratified)", "F1"].item()
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=30)
    ax.axhline(dummy_f1, ls=":", c="grey", lw=1)
g.figure.suptitle("Test metrics (dotted = stratified-dummy F1)", y=1.05)
savefig("04_test_metrics"); plt.show()

/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/1559083898.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  savefig("04_test_metrics"); plt.show()


In [15]:
fig, axes = plt.subplots(3, 3, figsize=(13, 12))
for i, model_name in enumerate(["Logistic Regression", "Naive Bayes", "Random Forest"]):
    for j, fs in enumerate(["text", "structural", "combined"]):
        m = fitted[(model_name, fs)]
        ConfusionMatrixDisplay.from_predictions(
            y_test, m.predict(X_test), display_labels=["Low", "High"],
            cmap="Blues", ax=axes[i, j], colorbar=False)
        axes[i, j].set_title(f"{model_name}\n{fs} (F1={f1_score(y_test, m.predict(X_test)):.3f})", fontsize=10)
plt.tight_layout(); savefig("05_confusion_matrices"); plt.show()

/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/2683253890.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("05_confusion_matrices"); plt.show()


In [16]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for model_name in ["Logistic Regression", "Naive Bayes", "Random Forest"]:
    RocCurveDisplay.from_estimator(fitted[(model_name, "combined")], X_test, y_test, ax=ax, name=model_name)
ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1, label="Chance")
ax.set_title("ROC – combined features"); ax.legend(loc="lower right")
plt.tight_layout(); savefig("06_roc_combined"); plt.show()

best_row = results[~results.Model.str.startswith("Dummy")].sort_values("F1", ascending=False).iloc[0]
best_key = (best_row.Model, best_row.Features)
best_model = fitted[best_key]
print(f"Best by test F1: {best_key}\n")
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=["Low quality", "High quality"], digits=3))

Best by test F1: ('Logistic Regression', 'combined')

              precision    recall  f1-score   support

 Low quality      0.853     0.608     0.710      1121
High quality      0.562     0.827     0.669       682

    accuracy                          0.691      1803
   macro avg      0.707     0.718     0.690      1803
weighted avg      0.743     0.691     0.695      1803



/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/1634512299.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("06_roc_combined"); plt.show()


## 7. Comparative analysis & interpretation


In [17]:
pivot_f1 = results[~results.Model.str.startswith("Dummy")].pivot(
    index="Model", columns="Features", values="F1")[["text", "structural", "combined"]]
pivot_auc = results[~results.Model.str.startswith("Dummy")].pivot(
    index="Model", columns="Features", values="ROC-AUC")[["text", "structural", "combined"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="Greens", ax=axes[0], cbar=False)
axes[0].set_title("Test F1 (high-quality)")
sns.heatmap(pivot_auc, annot=True, fmt=".3f", cmap="Blues", ax=axes[1], cbar=False)
axes[1].set_title("Test ROC-AUC")
for ax in axes:
    ax.set(xlabel="Feature set", ylabel="")
plt.tight_layout(); savefig("07_featureset_heatmap"); plt.show()

/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/3200551084.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("07_featureset_heatmap"); plt.show()


In [18]:
lr = fitted[("Logistic Regression", "combined")]
feat_names = lr.named_steps["features"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["clf"].coef_.ravel(), index=feat_names)
coefs.index = (coefs.index.str.replace("title_tfidf__", "[title] ")
                          .str.replace("body_tfidf__", "[body] ")
                          .str.replace("numeric__", "[struct] ")
                          .str.replace("tags__", "[tag] "))

top = pd.concat([coefs.nsmallest(15), coefs.nlargest(15).iloc[::-1]])
fig, ax = plt.subplots(figsize=(9, 9))
top.sort_values().plot.barh(ax=ax, color=["#C44E52" if v < 0 else "#55A868" for v in top.sort_values()])
ax.set(title="LR (combined) – top coefficients", xlabel="Coefficient")
plt.tight_layout(); savefig("08_lr_coefficients"); plt.show()

struct_coefs = coefs[coefs.index.str.startswith("[struct]")].sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
struct_coefs.plot.barh(ax=ax, color=["#C44E52" if v < 0 else "#55A868" for v in struct_coefs])
ax.set(title="LR structural coefficients", xlabel="Coefficient")
plt.tight_layout(); savefig("09_lr_structural"); plt.show()

/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/399766275.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("08_lr_coefficients"); plt.show()
/var/folders/62/n1qkxcl902x6krpqwsfyd4vh0000gn/T/ipykernel_6067/399766275.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig("09_lr_structural"); plt.show()


In [19]:
# Error analysis on best model
proba = best_model.predict_proba(X_test)[:, 1]
err = df.loc[X_test.index, ["Title", "Tags", "Score", "body_words", "n_code_blocks"]].assign(
    p_high=proba, true=y_test)
print("Confident FALSE POSITIVES (pred high, actually low):")
display(err[err.true == 0].nlargest(5, "p_high").round(3))
print("Confident FALSE NEGATIVES (pred low, actually high):")
display(err[err.true == 1].nsmallest(5, "p_high").round(3))

Confident FALSE POSITIVES (pred high, actually low):


,Title,Tags,Score,body_words,n_code_blocks,p_high,true
1006,Implementing generic polymorphism in C using union-pointer casting: approach...,|c|pointers|union|undefined-behavior|strict-aliasing|,0,173,1,0.860,0
1930,How does it follow that the implementation guarantees that less than `FOPEN_...,|c|language-lawyer|stdio|,0,96,1,0.860,0
1007,"In the C expression x = x + 1, does the left-hand sub-expression x undergo e...",|c|language-lawyer|,0,106,1,0.840,0
1946,"How to fix this error ""Failed to apply plugin 'dev.flutter.flutter-gradle-pl...",|java|android|flutter|kotlin|gradle|,0,86,6,0.835,0
3538,How to incorporate a variable in an xlookup formula in Excel VBA coding to p...,|excel|vba|,0,288,2,0.818,0


Confident FALSE NEGATIVES (pred low, actually high):


,Title,Tags,Score,body_words,n_code_blocks,p_high,true
8159,Need help registering to use APIs,|imgur|advice|,1,72,0,0.051,1
2333,Can an iOS app rely on iCloud Drive to finish syncing files after the app is...,|advice|ios|swift|cloudkit|ckasset|,1,101,0,0.053,1
4416,Count the number of vowels in a text,|ruby|advice|,1,48,1,0.076,1
261,Managing custom libraries with NASM or x86 - how to avoid importing methods ...,|advice|x86|nasm|,4,269,0,0.076,1
2801,"How can I build an AI automation workflow that extracts email data, stores i...",|python|artificial-intelligence|crm|erp|best-practices|,1,218,0,0.077,1


## 8. Summary


In [20]:
summary = results.copy()
summary["Config"] = summary.Model + " | " + summary.Features
summary = summary.set_index("Config")[["Accuracy", "Precision", "Recall", "F1", "Macro F1", "ROC-AUC"]]
display(summary.sort_values("F1", ascending=False).round(3))

clean = results[~results.Model.str.startswith("Dummy")]
best_by_fs = clean.loc[clean.groupby("Features").F1.idxmax(), ["Features", "Model", "F1", "ROC-AUC"]]
print("Best model per feature set:")
display(best_by_fs.round(3))
dummy_f1 = results.loc[results.Model == "Dummy (stratified)", "F1"].item()
print(f"Best F1 = {clean.F1.max():.3f}  vs stratified-dummy F1 = {dummy_f1:.3f}  (+{clean.F1.max() - dummy_f1:.3f})")
print("Saved figures:", sorted(p.name for p in FIG_DIR.glob("*.png")))
print("Saved results:", sorted(p.name for p in RES_DIR.glob("*.csv")))

,Accuracy,Precision,Recall,F1,Macro F1,ROC-AUC
Config,,,,,,
Logistic Regression | combined,0.691,0.562,0.827,0.669,0.690,0.795
Logistic Regression | structural,0.676,0.547,0.837,0.662,0.676,0.791
Random Forest | structural,0.673,0.546,0.801,0.649,0.671,0.765
Random Forest | combined,0.654,0.532,0.710,0.608,0.650,0.742
Naive Bayes | structural,0.659,0.539,0.685,0.603,0.653,0.721
Naive Bayes | combined,0.664,0.551,0.610,0.579,0.650,0.732
Logistic Regression | text,0.624,0.503,0.562,0.530,0.608,0.674
Naive Bayes | text,0.615,0.491,0.523,0.507,0.595,0.649
Random Forest | text,0.642,0.530,0.479,0.503,0.612,0.667


Best model per feature set:


,Features,Model,F1,ROC-AUC
8,combined,Logistic Regression,0.669,0.795
5,structural,Logistic Regression,0.662,0.791
2,text,Logistic Regression,0.530,0.674


Best F1 = 0.669  vs stratified-dummy F1 = 0.364  (+0.306)
Saved figures: ['01_score_distribution.png', '02_class_balance.png', '03_structural_by_class.png', '04_test_metrics.png', '05_confusion_matrices.png', '06_roc_combined.png', '07_featureset_heatmap.png', '08_lr_coefficients.png', '09_lr_structural.png']
Saved results: ['cv_results.csv', 'test_results.csv']


## 9. Conclusions

- The notebook builds a full **question-quality classifier** from Stack Overflow posts available at posting time.
- Expect **structural + tag features** to usually beat pure TF-IDF text; combining both is often best or near-best.
- **Logistic Regression** is a strong, interpretable baseline for this sparse/high-dimensional setting.
- Performance is **better than chance**, but moderate — quality also depends on community dynamics and question type, not only writing style.
- Do **not** use post-hoc engagement or extraction-time reputation as features; they leak the label.

### Outputs
- `figures/` — plots for the report  
- `results/cv_results.csv`, `results/test_results.csv` — metrics tables
